# 00 · Chuẩn bị: lab HeteroES và các máy GPU

Notebook này đi qua những gì cần có **trước** khi chạy một thí nghiệm, theo đúng thứ tự một nhóm nghiên cứu làm:

| Vai trò | Làm gì | Ở đâu |
|---|---|---|
| Người nghiên cứu | đăng nhập lab, xem các GPU đang có | notebook này (mục 1–2) |
| Chủ máy GPU (người cho mượn) | cài worker bằng một dòng lệnh, nhập mã ghép | terminal trên máy GPU (mục 3) |
| Quản trị lab | duyệt máy mới, bật/tắt đóng góp | trang Workers của lab, hoặc notebook này (mục 4) |
| Người nghiên cứu | chọn mô hình, dữ liệu, bộ chấm, cấu hình rồi chạy | các notebook E1–E5 |

Các thí nghiệm của đề tài dùng 4 GPU của 4 máy khác nhau, nối qua Tailscale: hai RTX 5070 Ti (máy của Nhân, kiêm
coordinator, và máy của Nhựt) và hai RTX 3060 (máy của Quang và của Thảo, chạy trong WSL2 trên Windows).

## 1. Đăng nhập lab

Lab công khai ở `https://lab.heteroes.site`. Đăng nhập một lần trong terminal (mật khẩu không bao giờ được ghi vào
repo hay notebook; phiên có hạn 12 giờ và được lưu ở `~/.config/heteroes-demo/session.json`, quyền 0600):

```bash
python3 scripts/lab-login.py            # hỏi tên và mật khẩu; chạy lại khi phiên hết hạn
```

Một notebook đang chạy dài sẽ tự đọc lại phiên mới khi phiên cũ hết hạn.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
SRC = (HERE / ".." / ".." / "src").resolve()
for path in (str(SRC), str(HERE)):
    if path not in sys.path:
        sys.path.insert(0, path)

from heteroes.sdk_lab import Lab  # noqa: E402

lab = Lab.from_session()
print(lab)

Lab('https://lab.heteroes.site', workspace='lab')


## 2. Các GPU của lab

Mỗi máy GPU là một *worker*. `online` = máy đang kết nối với lab; `ON` = chủ máy cho phép lab dùng GPU; `bận` = đang
làm một run hay một lần kiểm thử.

In [2]:
for d in lab.devices():
    print(f"{d.name:14} {d.gpu:28} {d.vram_gb:5.1f} GB  online={d.online!s:5} ON={d.on!s:5} bận={d.busy!s:5}"
          f"  coordinator={d.coordinator}")

5070ti-nhan    NVIDIA GeForce RTX 5070 Ti    15.9 GB  online=True  ON=True  bận=False  coordinator=True
3060-quang     NVIDIA GeForce RTX 3060       12.0 GB  online=True  ON=True  bận=True   coordinator=False
1660s          NVIDIA GeForce GTX 1660 SUPER   6.0 GB  online=False ON=False bận=False  coordinator=False
5070ti-nhut    NVIDIA GeForce RTX 5070 Ti    15.9 GB  online=True  ON=True  bận=False  coordinator=False
3060-thao      NVIDIA GeForce RTX 3060       12.0 GB  online=True  ON=True  bận=False  coordinator=False


## 3. Thêm một máy GPU (chủ máy làm)

Trên máy GPU (Linux, hoặc Ubuntu trong WSL2 trên Windows), dán một dòng lệnh. Lệnh này không cần quyền quản trị, tải bộ
cài từ lab và kiểm SHA-256 từng tệp:

```bash
curl -fsSL 'https://lab.heteroes.site/downloads/install-worker.sh' | HETEROES_DOWNLOADS='https://lab.heteroes.site/downloads' bash
```

Bộ cài kiểm tra driver và GPU, rồi hỏi **mã ghép** (pairing code). Mã ghép tạo ở trang Workers → *Add a GPU computer*,
hoặc bằng ô dưới (đặt `TAO_MA_GHEP = True`). Mã dùng một lần và hết hạn sau vài phút.

Sau khi ghép, chủ máy điều khiển máy của mình bằng `heteroes-worker`:

| Lệnh | Làm gì |
|---|---|
| `heteroes-worker status` | máy đang làm gì, lab nào, GPU, phiên bản runtime |
| `heteroes-worker on` / `off` | cho / thôi cho lab dùng GPU (run đang chạy tiếp trên các máy khác) |
| `heteroes-worker watch` | màn hình trực tiếp: run, thế hệ, GPU, log |
| `heteroes-worker doctor` | kiểm tra driver, ổ đĩa, dịch vụ, mạng, lab và đồng hồ |
| `heteroes-worker clean` | xoá bản tải và runtime cũ (không bao giờ xoá dữ liệu run) |

In [3]:
TAO_MA_GHEP = False  # True: tạo một mã ghép mới cho một máy GPU

if TAO_MA_GHEP:
    reply = lab.request("POST", "/api/v1/devices/pairing-codes", {})
    print("mã ghép:", reply["code"], f"(hết hạn sau {reply.get('expiresInSeconds', '?')} giây)")

## 4. Duyệt máy mới và bật đóng góp (quản trị lab)

Một máy vừa ghép ở trạng thái `PENDING`: lab chưa giao việc cho nó cho đến khi quản trị viên duyệt. Ô dưới liệt kê máy
chờ duyệt; đặt `DUYET = True` để duyệt tất cả.

In [4]:
DUYET = False

for d in lab.request("GET", "/api/v1/platform/devices"):
    print(f"{d['displayName']:14} {d['approvalStatus']:9} ON={d['contributionEnabled']}")
    if DUYET and d["approvalStatus"] == "PENDING":
        lab.request("PATCH", f"/api/v1/platform/devices/{d['deviceId']}", {"approved": True, "workspaceId": lab.workspace})
        print("   → đã duyệt")

5070ti-nhan    APPROVED  ON=True
3060-quang     APPROVED  ON=True
1660s          APPROVED  ON=False
5070ti-nhut    APPROVED  ON=True
3060-thao      APPROVED  ON=True


## 5. Lab kiểm thử từng GPU với từng mô hình

Trước run đầu tiên của một mô hình trên một GPU, lab tự **kiểm thử** (qualify) cặp đó: GPU tải mô hình (từ Hugging
Face, hoặc từ lab với mô hình đã train), chạy một phép ES rất nhỏ bằng bộ suy luận nghiêm ngặt và so SHA-256 kết quả
với máy tham chiếu. Lần kiểm thử cũng đo bộ nhớ của từng cỡ lô, để lab gợi ý cỡ lô cho mỗi GPU. GPU không đạt thì không
được dùng cho mô hình đó. Các notebook E1–E5 gọi bước này tự động khi gửi run.

Lỗi hay gặp:

| Hiện tượng | Nguyên nhân | Cách xử lý |
|---|---|---|
| `GPU_UNSUPPORTED_BY_DECODER` | GPU cũ hơn RTX 30 (ví dụ GTX 1660, sm75): bộ suy luận nghiêm ngặt dùng lệnh nhân ma trận số nguyên của sm80 | dùng GPU từ RTX 30 trở lên |
| Máy WSL nhận việc nhưng không gửi được kết quả | MTU mạng của WSL nhỏ (1280) nên gói Tailscale lớn bị rớt | trong WSL: `sudo ip link set tailscale0 mtu 1200` |
| Ổ C: của Windows đầy | ổ ảo `ext4.vhdx` của WSL nằm trên C: và không tự co lại | `heteroes-worker clean`; nén vhdx bằng `wsl --shutdown` rồi `diskpart` |
| `401` sau 12 giờ | phiên lab hết hạn | chạy lại `scripts/lab-login.py` |

In [5]:
import json

for m in lab.request("GET", "/api/v1/models"):
    print(f"{m['name'][:60]:60} {m.get('status')}")

Qwen/Qwen2.5-0.5B-Instruct                                   REGISTERED
HuggingFaceTB/SmolLM2-360M-Instruct                          REGISTERED
Qwen/Qwen2.5-0.5B-Instruct                                   REGISTERED
HuggingFaceTB/SmolLM2-360M-Instruct+e2-arith-smollm2-s01     READY
Qwen/Qwen2.5-1.5B-Instruct                                   REGISTERED
Qwen/Qwen2.5-1.5B-Instruct+e4-countdown-qwen15-s01           READY


Tiếp theo: mở một notebook thí nghiệm (E1–E5). Mỗi notebook đi qua dữ liệu → mô hình → bộ chấm → cấu hình → chạy
trên lab → tốc độ → quá trình học → chấm test → ví dụ trước/sau.